# LightGBM Training Metrics - Notebook 表示確認

メトリクスが Notebook セル出力に `println()` で表示されることを確認します。

In [1]:
import os, glob, findspark

os.environ['SPARK_HOME'] = '/opt/spark'
os.environ['PYSPARK_PYTHON'] = '/usr/bin/python3'
findspark.init()

# 最新の修正済み JAR を自動選択
jars = sorted(glob.glob('/opt/jars/synapseml-lightgbm_2.12-*.jar'))
MODIFIED_JAR = jars[-1]
print(f'Using JAR: {MODIFIED_JAR}')

Using JAR: /opt/jars/synapseml-lightgbm_2.12-HEAD-20260215-1039.jar


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('LightGBM-Metrics-Notebook')
    .master('local[*]')
    .config('spark.jars', MODIFIED_JAR)
    .config('spark.jars.packages', 'com.microsoft.azure:synapseml-lightgbm_2.12:1.0.8')
    .config('spark.jars.ivy', '/tmp/.ivy2')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
print(f'Spark version: {spark.version}')

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
com.microsoft.azure#synapseml-lightgbm_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4962cc04-d0df-4e2f-a4a0-0180acdfec19;1.0
	confs: [default]
	found com.microsoft.azure#synapseml-lightgbm_2.12;1.0.8 in central
	found com.microsoft.azure#synapseml-core_2.12;1.0.8 in central
	found org.apache.spark#spark-avro_2.12;3.4.1 in central
	found org.tukaani#xz;1.9 in central
	found commons-lang#commons-lang;2.6 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central
	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 i

Spark version: 3.5.8


In [3]:
import pyspark.sql.functions as F

df = spark.read.parquet('/opt/data/toctou_test')
total = df.count()
print(f'Total rows: {total:,}')

# Train/Test split
df_train, df_test = df.randomSplit([0.8, 0.2], seed=42)
df_train = df_train.withColumn('isValidation', F.rand(seed=42) < 0.1)
print(f'Train: {df_train.count():,}  Test: {df_test.count():,}')

Total rows: 100,000


Train: 80,241  Test: 19,759


In [4]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from synapse.ml.lightgbm import LightGBMClassifier

cat_cols = ['cat_region', 'cat_plan', 'cat_channel', 'cat_product', 'cat_zipcode']
num_cols = ['num_income', 'num_age', 'num_tenure', 'num_usage', 'num_score', 'num_noise']

indexers = [StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep') for c in cat_cols]
cat_idx_cols = [f'{c}_idx' for c in cat_cols]
feature_cols = num_cols + cat_idx_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='skip')

lgbm = (
    LightGBMClassifier()
    .setLabelCol('label')
    .setFeaturesCol('features')
    .setObjective('binary')
    .setNumLeaves(63)
    .setNumIterations(200)
    .setLearningRate(0.05)
    .setMinDataInLeaf(100)
    .setFeatureFraction(0.8)
    .setBaggingFraction(0.8)
    .setBaggingFreq(1)
    .setCategoricalSlotNames(cat_idx_cols)
    .setVerbosity(1)
    .setIsUnbalance(True)
    .setUseSingleDatasetMode(False)
    .setIsProvideTrainingMetric(True)
    .setMetric('binary_logloss,average_precision')
    .setValidationIndicatorCol('isValidation')
    .setEarlyStoppingRound(20)
)

pipeline = Pipeline(stages=indexers + [assembler, lgbm])
print('Pipeline ready')

Pipeline ready


## Training

以下のセルを実行すると、セル出力に `LightGBM training metrics:` ヘッダーに続いて
`[0]`, `[10]`, `[20]`, ... の形式でメトリクスが表示されるはずです。

Early stopping が発動した場合はその行も表示されます。

In [5]:
import time

t0 = time.time()
model = pipeline.fit(df_train)
elapsed = time.time() - t0
print(f'\nTraining done ({elapsed:.1f}s)')

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Saving data reference to binary buffer
[LightGBM] [Info] Loaded reference dataset: 11 features, 72305 num_data


[Stage 32:====================================================>   (15 + 1) / 16]

[LightGBM] [Info] Number of positive: 6016, number of negative: 66289
[LightGBM] [Warning] Only find one worker, will switch to serial tree learner
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013334 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2423
[LightGBM] [Info] Number of data points in the train set: 72305, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.083203 -> initscore=-2.399601
[LightGBM] [Info] Start training from score -2.399601
LightGBM training metrics:
[0]	train's binary_logloss: 0.274868	train's average_precision: 0.196160	valid's binary_logloss: 0.271284	valid's average_precision: 0.148414
[10]	train's binary_logloss: 0.272175	train's average_precision: 0.538764	valid's binary_logloss: 0.302978	valid's average_precision: 0.191106
[20]	train's binary_logloss: 0.286141	train's average_precision: 0.661351	valid's binary_logloss: 0.334062	valid's

In [6]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

df_pred = model.transform(df_test)

auc = BinaryClassificationEvaluator(
    labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
).evaluate(df_pred)

accuracy = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
).evaluate(df_pred)

print(f'AUC-ROC  : {auc:.4f}')
print(f'Accuracy : {accuracy:.4f}')

AUC-ROC  : 0.7494
Accuracy : 0.8570


## Training Metric History (構造化データ)

`_java_obj` 経由で JVM のアクセサメソッドを呼び、各イテレーションのメトリクス値を取得します。

In [ ]:
spark.stop()

In [15]:
import pandas as pd

lgbm_model = model.stages[-1]
java_model = lgbm_model._java_obj

# JVM アクセサメソッドで取得
iterations = list(java_model.getTrainingMetricIterations())
metric_names = list(java_model.getTrainingMetricNames())
print(f"Recorded iterations: {len(iterations)}")
print(f"Metric names: {metric_names}")

# pandas DataFrame に変換
data = {"iteration": iterations}
for name in metric_names:
    train_vals = list(java_model.getTrainingMetricTrainValues(name))
    valid_vals = list(java_model.getTrainingMetricValidValues(name))
    # NaN でなければ列に追加
    if not all(v != v for v in train_vals):  # NaN check
        data[f"train_{name}"] = train_vals
    if not all(v != v for v in valid_vals):
        data[f"valid_{name}"] = valid_vals

df_metrics = pd.DataFrame(data)
df_metrics

Recorded iterations: 21
Metric names: ['binary_logloss', 'average_precision']


,iteration,train_binary_logloss,valid_binary_logloss,train_average_precision,valid_average_precision
0,0,0.275783,0.266148,0.195970,0.136434
1,1,0.271732,0.267068,0.267978,0.163062
2,2,0.268310,0.269035,0.311739,0.171440
3,3,0.267091,0.272045,0.353685,0.173774
4,4,0.268094,0.276319,0.377809,0.179143
5,5,0.268187,0.280661,0.400773,0.174456
6,6,0.269312,0.284153,0.423018,0.185932
7,7,0.270413,0.288132,0.446604,0.188119
8,8,0.271301,0.292318,0.470822,0.189329
9,9,0.272430,0.295897,0.493746,0.188638
